# v9_b Step 3b: Match Step 3 khipu features and authentic standards to Thyroid features

This Thyroid-only notebook follows the matching logic of `v8_step_3b_feature_matching_1016.ipynb`. It reads the four v9_b Step 3 `*_features.json` files, the updated local authentic-standard list, and the two local Thyroid full feature tables. It retains all matches within m/z <10 ppm and RT difference <15 s, after subtracting 120 s from DnHz target RT. It does not filter by sample intensity or evaluate standard MS2.

In [ ]:
import os
import json
import pandas as pd
from asari.tools.file_io import read_features_from_asari_table
from asari.tools import match_features as mf
from pathlib import Path
ROOT = Path.cwd().resolve()
assert (ROOT / 'results/step3_khipus').is_dir(), 'Run this notebook from the v9_b directory'


In [ ]:

def write_matched_features_to_json(matched, dict_query, dict_features, output_file):
    matched_features = []
    for feature_id, matched_list in matched.items():
        matched_feature = dict_query.get(feature_id)
        if matched_feature:
            matched_features.append({
                'Feature': matched_feature,
                'Matched_libs': [dict_features[lib_id] for lib_id in matched_list if lib_id in dict_features]
            })
    
    with open(output_file, 'w') as f:
        json.dump(matched_features, f, indent=4)

## Load v9_b Step 3 khipu features

In [ ]:
candidates_dir = str(ROOT / 'results/step3_khipus')
DnCl_candidates = json.load(open(os.path.join(candidates_dir, 'DnCl_pellets_features.json'))) + json.load(
    open(os.path.join(candidates_dir, 'DnCl_supernatant_features.json')))

DnHz_candidates = json.load(open(os.path.join(candidates_dir, 'DnHz_pellets_features.json'))) + json.load(
    open(os.path.join(candidates_dir, 'DnHz_supernatant_features.json')))

# Keep both sources when they contain the same feature ID.
from collections import Counter
for candidates in (DnCl_candidates, DnHz_candidates):
    duplicate_ids = {feature_id for feature_id, count in Counter(f['id'] for f in candidates).items() if count > 1}
    for feature in candidates:
        if feature['id'] in duplicate_ids:
            feature['original_id'] = feature['id']
            feature['id'] = f"{feature['source']}:{feature['id']}"

print(f"Read {len(DnCl_candidates)} DnCl candidates and {len(DnHz_candidates)} DnHz candidates from {candidates_dir}")

## Expand authentic-standard RT entries

In [ ]:
# SZ manual annotation 
manual_annotation_path = str(ROOT / 'input/step3b_thyroid/Authentic_standard_list.csv')
manu_DnCl, manu_DnHz = [], []
for index, row in pd.read_csv(manual_annotation_path).iterrows():
    # No, Derivatization, Name, KEGG_number, derivatized [M+H]+ mass, RT_min
    feature_id = row['No']
    mz = row['Exact_mass_[M+H]+_after_derivatization']
    rtList = row['RT_min'].split(';')
    lib_id = row['KEGG_number']
    method = row['Derivatization']
    if 'DnCl' in method:
        for rt_index, rt in enumerate(rtList, start=1):
            rt_min = float(rt)
            manu_DnCl.append({'id': f'STD:{feature_id}:RT{rt_index}', 'standard_id': feature_id,
                            'standard_name': row['Name'], 'standard_rt_min': rt_min,
                            'mz': float(mz), 'rtime': rt_min * 60, 'lib_id': lib_id})
    elif 'DnHz' in method:
        for rt_index, rt in enumerate(rtList, start=1):
            rt_min = float(rt)
            manu_DnHz.append({'id': f'STD:{feature_id}:RT{rt_index}', 'standard_id': feature_id,
                            'standard_name': row['Name'], 'standard_rt_min': rt_min,
                            'mz': float(mz), 'rtime': rt_min * 60, 'lib_id': lib_id})

manu_DnCl[1]

## Match v9_b queries to Thyroid features

In [ ]:
outdir = str(ROOT / 'results/step3b_thyroid_feature_matching') + '/'
os.mkdir(outdir) if not os.path.exists(outdir) else None

other_dir = str(ROOT / 'input/step3b_thyroid') + '/'
celi_DnCl = f'{other_dir}Thyroid/DnCl_full_Feature_table.tsv'
celi_DnHz = f'{other_dir}Thyroid/DnHz_full_Feature_table.tsv'

num_samples, celi_DnCl_list_features = read_features_from_asari_table(open(celi_DnCl).read())
num_samples, celi_DnHz_list_features = read_features_from_asari_table(open(celi_DnHz).read())

# Align DnHz RT to the candidate tables, retaining the original target RT.
for feature in celi_DnHz_list_features:
    feature['rtime_raw_s'] = feature['rtime']
    feature['rtime'] -= 120

matched = mf.list_match_lcms_features(DnCl_candidates + manu_DnCl, 
                                      celi_DnCl_list_features,  mz_ppm=10,  rt_tolerance=15)
dict_features = {f['id']: f for f in celi_DnCl_list_features}
fdict_DnCl_candidates = {f['id']: f for f in DnCl_candidates + manu_DnCl}
write_matched_features_to_json(matched, fdict_DnCl_candidates, dict_features, 
                               outdir + 'DnCl_matched_celi_DnCl.json')

matched = mf.list_match_lcms_features(DnHz_candidates + manu_DnHz, 
                                      celi_DnHz_list_features,  mz_ppm=10,  rt_tolerance=15)
dict_features = {f['id']: f for f in celi_DnHz_list_features}
fdict_DnHz_candidates = {f['id']: f for f in DnHz_candidates + manu_DnHz}
write_matched_features_to_json(matched, fdict_DnHz_candidates, dict_features,
                              outdir + 'DnHz_matched_celi_DnHz.json')

## Reverse match: Thyroid features to v9_b queries

In [ ]:
# Reverse matching: match celi features to candidates
matched = mf.list_match_lcms_features(celi_DnCl_list_features, DnCl_candidates + manu_DnCl, 
                                       mz_ppm=10,  rt_tolerance=15)
dict_features = {f['id']: f for f in celi_DnCl_list_features}
fdict_DnCl_candidates = {f['id']: f for f in DnCl_candidates + manu_DnCl}
write_matched_features_to_json(matched, dict_features, fdict_DnCl_candidates, 
                               outdir + 'Celi_DnCl_matched_features.json')

matched = mf.list_match_lcms_features(celi_DnHz_list_features, DnHz_candidates + manu_DnHz, 
                                      mz_ppm=10,  rt_tolerance=15)
dict_features = {f['id']: f for f in celi_DnHz_list_features}
fdict_DnHz_candidates = {f['id']: f for f in DnHz_candidates + manu_DnHz}
write_matched_features_to_json(matched, dict_features, fdict_DnHz_candidates,
                              outdir + 'Celi_DnHz_matched_features.json')

## Check the four outputs

In [ ]:
from pathlib import Path
for name in ('DnCl_matched_celi_DnCl.json', 'DnHz_matched_celi_DnHz.json',
             'Celi_DnCl_matched_features.json', 'Celi_DnHz_matched_features.json'):
    path = Path(outdir) / name
    records = json.load(path.open())
    print(name, 'query_records=', len(records), 'matched_pairs=', sum(len(row['Matched_libs']) for row in records))